# 30 — Embedding Evaluation

Thin notebook: it only **imports**, **calls** `src/embedding_metrics.py` + `src/embedding_report.py`, and **displays**.
This is stage 3 of the pipeline, and the only notebook in this pillar. It computes the metrics, draws the figures and writes the report.

**Input** comes from `eVTOL-Embedding-Extraction` (stage 2) under `paths.pipeline_root`:
- the figure sets, from notebook `20_figure_selection`;
- the embeddings, from notebook `22_embedding_extraction`.

**Output:**
- numbers in `paths.metrics_dir/<tag>/<set>/`;
- the report in `docs/embedding_evaluation/embedding_evaluation_report.md`, with its figures in `docs/embedding_evaluation/figs/`.

It covers only the embedding part of the supervisor report and uses no taxonomy labels yet:

| Section | Question | Report § |
|---|---|---|
| 0 | Which figures went in, and why? (selection funnel, figure sets, coverage) | — |
| A | Are the vectors technically sound? (NaN/Inf, duplicates, dead dims, cosine spread) | §2 |
| B | Is there real structure beyond random noise? (PC1 vs random, effective dim, Hopkins) | §3 |
| C | Do clusters form without labels, and are they stable? (silhouette, HDBSCAN, bootstrap ARI) | §1.3 / §4.2 |
| D | Which layer × pooling ranks best, and why mean_patch vs cls? | §1.3 |

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'config.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Markdown
from src.config_loader import load_config
from src import embedding_metrics as em, embedding_report as er

cfg = load_config()
sets = em.load_sets(cfg)
print('sets:', {k: len(v) for k, v in sets.items()})
print('embeddings:', cfg['metrics']['embeddings'], '| scope:', cfg['metrics']['aircraft_scope'])

## 0. What went in
This section reads the selection that notebook 20 saved and does not recompute it.

In [ ]:
sel = er.load_selection(cfg)
display(sel['funnel'])
pd.DataFrame(sel['summary']['sets']).T

## 1. Metrics for every embedding run × set
This takes about 20 s per set, mostly for the bootstrap refits. Runs whose embeddings don't exist yet are skipped.

In [ ]:
comparison = em.run_all(cfg, sets)
if len(comparison):
    best = comparison.sort_values('rank_sum', ascending=False).groupby(['embedding', 'set']).head(1)
    display(best[['embedding', 'set', 'n_figures', 'n_aircraft', 'layer', 'pooling',
                  'pc1_ratio_vs_random', 'hopkins', 'best_kmeans_silhouette',
                  'bootstrap_ari', 'hdbscan_noise_frac']].sort_values(['embedding', 'set']))

## 2. Figures and report
Sections A–D are written in full for `report.detail_embedding` × `report.detail_set`.

In [ ]:
rc = cfg['report']
m = er.load_metrics(cfg, rc['detail_embedding'], rc['detail_set'])
figs = er.draw_figures(m, cfg)
path = er.write_report(m, cfg, sel=sel, comp=er.load_comparison(cfg))
print(len(figs), 'figures |', path)
display(Markdown(path.read_text(encoding='utf-8')))